## Create optimisation objectives based on existing annotations

This template loops through the features of each record, and creates a corresponding optimisation objective for features passing certain conditions.
This is useful for specifying constraints or objectives for only a section of the record (for example, adding an optimisation objective at the @cds annotations), or for objectives that cannot be specified through the Genbank API (for example, custom codon tables).

In [ ]:
import csv
import os

import Bio

import dnachisel
import dnacauldron

In [ ]:
dir_to_sculpt = "n1_raw_gb/"

In [ ]:
records_to_sculpt = dnacauldron.biotools.load_records_from_files(folder=dir_to_sculpt, use_file_names_as_ids=True)

In [ ]:
all_constraints=[
        dnachisel.AvoidPattern("BsaI_site"),
        dnachisel.AvoidPattern("NotI_site"),
        dnachisel.AvoidPattern("XbaI_site"),
        dnachisel.AvoidPattern("ClaI_site"),
        dnachisel.AvoidPattern("8x1mer"),
        dnachisel.builtin_specifications.UniquifyAllKmers(15),
        dnachisel.EnforceGCContent(mini=0.1, maxi=0.9, window=50),
#         dnachisel.EnforceTranslation(),
]

In [ ]:
# Downloaded from https://github.com/Edinburgh-Genome-Foundry/codon-usage-tables/blob/master/codon_usage_data/tables/c_elegans_6239.csv
codon_table_csv_file = "n2_codon_table/c_elegans_6239.csv"

In [ ]:
with open(codon_table_csv_file, 'r') as file:
    csv_string = file.read()

In [ ]:
import python_codon_tables as pct
rna_table = pct.csv_string_to_codons_dict(csv_string)
codon_table = pct.python_codon_tables.table_with_U_replaced_by_T(rna_table)

In [ ]:
record_objective_dict = {}
for record in records_to_sculpt:
    objectives = []
    for feature in record.features:
        try:
            if "@cds" in feature.qualifiers["label"]:  # specify condition
                if not len(feature) % 3 == 0:
                    print("CDS annotation is not multiple of 3:", record.id)
                # Specify objective:
                objective = dnachisel.builtin_specifications.CodonOptimize(method="use_best_codon", location=feature.location, codon_usage_table=codon_table)
                objectives.append(objective)
        except:
            pass

    record_objective_dict[record.id] = objectives

In [ ]:
target_dir = "n3_sculpted/"

In [ ]:
try:
    os.mkdir(target_dir)
except:
    pass

In [ ]:
for record in records_to_sculpt:
    problem = dnachisel.DnaOptimizationProblem.from_record(record,
                                                           extra_constraints=all_constraints,
                                                           extra_objectives=record_objective_dict[record.id])
    print(record.id)
    len_seq = len(problem.sequence)
    print("Length:", len_seq)
    target_path = os.path.join(target_dir, record.id + ".zip")
    problem.optimize_with_report(target=target_path)
    print()
    print(problem.constraints_text_summary())
    print(problem.objectives_text_summary())
    print()